# XGB One-Versus-Rest — v002 (4GB VRAM safe)
Improvements over v001:
- Original data included in training (was loaded but never trained on)
- All FE applied to `orig` (digit, threshold, logit, ratio features)
- StratifiedKFold instead of plain KFold
- Ratio / interaction features between main numeric drivers
- `max_bin=256` (safe for 4GB RTX 3050; was 10000 = OOM)
- Aggressive GPU memory flush between each binary model
- 500 Optuna trials (was 200)


In [ ]:
NAME = '002'

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import seaborn as sns

In [ ]:
%load_ext cudf.pandas

import numpy as np, pandas as pd, gc
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 200)

In [ ]:
train_file    = '/kaggle/input/competitions/playground-series-s6e4/train.csv'
test_file     = '/kaggle/input/competitions/playground-series-s6e4/test.csv'
original_file = '/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv'

train = pd.read_csv(train_file)
test  = pd.read_csv(test_file)
orig  = pd.read_csv(original_file)

train_ids = train['id'].copy()
test_ids  = test['id'].copy()
print(f"train: {train.shape}, test: {test.shape}, orig: {orig.shape}")

In [ ]:
train['Irrigation_Need'] = train['Irrigation_Need'].map(_classes := {'Low': 0, 'Medium': 1, 'High': 2})
orig['Irrigation_Need']  = orig['Irrigation_Need'].map(_classes)

In [ ]:
TARGET = 'Irrigation_Need'
NUMS = list(train.drop(['id', TARGET], axis=1)._get_numeric_data().columns)
CATS = [c for c in train.drop(['id', TARGET], axis=1).columns if c not in NUMS]
assert len(NUMS) + len(CATS) + 2 == train.shape[1]
print(f"Cats ({len(CATS)}): {CATS}")
print(f"Nums ({len(NUMS)}): {NUMS}")

# Feature Engineering

In [ ]:
NEW_NUMS    = []
NEW_CATS    = []
NUM_AS_CAT  = []
TO_REMOVE   = []
NON_TE_CATS = []

In [ ]:
# 2-way categorical combos (3-way disabled — GPU OOM on 4GB VRAM)
_to_combo = CATS
for i, c1 in enumerate(_to_combo[:-1]):
    for j, c2 in enumerate(_to_combo[i + 1:]):
        _new_col = f'COMBO_{c1}_{c2}'
        for df in [train, test, orig]:
            df[_new_col] = df[c1].astype('str') + '_' + df[c2].astype('str')
        NEW_CATS.append(_new_col)
print(f"2-way combo cats: {len(NEW_CATS)}")

In [ ]:
# Frequency encoding from ALL data
for cat in CATS + NEW_CATS:
    freq = pd.concat([train[cat], orig[cat], test[cat]]).value_counts(normalize=True)
    for df in [train, test, orig]:
        df[f'FREQ_{cat}'] = df[cat].map(freq).fillna(0).astype('float32')
    NEW_NUMS.append(f'FREQ_{cat}')

In [ ]:
# Numerical as categorical
for col in NUMS:
    _new_col = f'CAT_{col}'
    NUM_AS_CAT.append(_new_col)
    for df in [train, test, orig]:
        df[_new_col] = df[col].astype(str).astype('category')

In [ ]:
# Digit features — applied to orig too so it can be used in training
# adapted from https://www.kaggle.com/code/yunsuxiaozi/pss6e4-xgb-cv-0-979805
M = train[NUMS].max()
DIGIT_FEATURES = []

for c in NUMS:
    for df in [train, test, orig]:
        for k in range(-4, 4):
            df[f"{c}_digit{k}"] = (df[c] // (10**k) % 10).astype('int8')
            DIGIT_FEATURES += [f"{c}_digit{k}"]
        if M[c] < 10:
            df[c] = df[c].round(3)
        elif M[c] < 100:
            df[c] = df[c].round(2)
        else:
            df[c] = df[c].round(1)

DIGIT_FEATURES = list(set(DIGIT_FEATURES))

DROP = [c for c in test.columns if test[c].nunique() == 1]
print(f"DROP: {DROP}")
for df in [train, test, orig]:
    df.drop([c for c in DROP if c in df.columns], axis=1, inplace=True)

DIGIT_FEATURES = list(set(DIGIT_FEATURES) - set(DROP))
NEW_CATS += DIGIT_FEATURES

see https://www.kaggle.com/code/cdeotte/original-data-exact-formula for exact formula

In [ ]:
# Threshold boolean features — applied to orig too
for df in [train, test, orig]:
    df["soil_lt_25"]  = (df["Soil_Moisture"] < 25).astype(int)
    df["temp_gt_30"]  = (df["Temperature_C"] > 30).astype(int)
    df["rain_lt_300"] = (df["Rainfall_mm"]    < 300).astype(int)
    df["wind_gt_10"]  = (df["Wind_Speed_kmh"] > 10).astype(int)

TRES_CATS = ['soil_lt_25', 'temp_gt_30', 'rain_lt_300', 'wind_gt_10']
NEW_CATS += TRES_CATS

In [ ]:
# Logit formula features — applied to orig too
for df_ in [train, test, orig]:
    df = pd.get_dummies(df_[NUMS + CATS + TRES_CATS], columns=CATS, drop_first=False)
    df_['logit(P(y=Low))']    = ( 16.3173
        + (-11.0237 * df["soil_lt_25"]) + (-5.8559 * df["temp_gt_30"])
        + (-10.8500 * df["rain_lt_300"]) + (-5.8284 * df["wind_gt_10"])
        + (-5.4155 * df["Crop_Growth_Stage_Flowering"]) + (5.5073 * df["Crop_Growth_Stage_Harvest"])
        + ( 5.2299 * df["Crop_Growth_Stage_Sowing"])   + (-5.4617 * df["Crop_Growth_Stage_Vegetative"])
        + (-3.0014 * df["Mulching_Used_No"])            + (2.8613  * df["Mulching_Used_Yes"]))
    df_['logit(P(y=Medium))'] = ( 4.6524
        + ( 0.3290 * df["soil_lt_25"]) + (-0.0204 * df["temp_gt_30"])
        + ( 0.1542 * df["rain_lt_300"]) + (0.0841 * df["wind_gt_10"])
        + ( 0.3586 * df["Crop_Growth_Stage_Flowering"]) + (-0.1348 * df["Crop_Growth_Stage_Harvest"])
        + (-0.3547 * df["Crop_Growth_Stage_Sowing"])    + ( 0.3334 * df["Crop_Growth_Stage_Vegetative"])
        + ( 0.1883 * df["Mulching_Used_No"])             + (0.0142 * df["Mulching_Used_Yes"]))
    df_['logit(P(y=High))']   = (-20.9697
        + (10.6947 * df["soil_lt_25"]) + (5.8763 * df["temp_gt_30"])
        + (10.6958 * df["rain_lt_300"]) + (5.7444 * df["wind_gt_10"])
        + ( 5.0569 * df["Crop_Growth_Stage_Flowering"]) + (-5.3725 * df["Crop_Growth_Stage_Harvest"])
        + (-4.8752 * df["Crop_Growth_Stage_Sowing"])    + ( 5.1283 * df["Crop_Growth_Stage_Vegetative"])
        + ( 2.8131 * df["Mulching_Used_No"])             + (-2.8755 * df["Mulching_Used_Yes"]))

NEW_NUMS += ['logit(P(y=Low))', 'logit(P(y=Medium))', 'logit(P(y=High))']

In [ ]:
train.isna().any().any(), test.isna().any().any(), orig.isna().any().any()

In [ ]:
# ORIG target stats merged into all three splits
for col in CATS + NUMS:
    stats = orig.groupby(col)[TARGET].agg(['mean', 'std']).reset_index()
    stats.columns = [col] + [f"ORIG_{col}_{s}" for s in ['mean', 'std']]
    fill_values = {f"ORIG_{col}_mean": 0.5, f"ORIG_{col}_std": 0}

    train = train.merge(stats, on=col, how='left').fillna(value=fill_values)
    test  = test.merge(stats,  on=col, how='left').fillna(value=fill_values)
    orig  = orig.merge(stats,  on=col, how='left').fillna(value=fill_values)
    NEW_NUMS.extend([f"ORIG_{col}_{s}" for s in ['mean', 'std']])

In [ ]:
# Ratio / interaction features between the 4 key numeric drivers
EPS = 1e-6
for df in [train, test, orig]:
    df['water_avail']    = df['Soil_Moisture'] + df['Rainfall_mm'] / 300.0
    df['heat_stress']    = df['Temperature_C'] / (df['Soil_Moisture'] + EPS)
    df['rain_cool']      = df['Rainfall_mm']   / (df['Temperature_C'] + EPS)
    df['wind_evap']      = df['Wind_Speed_kmh'] * df['Temperature_C'] / 100.0
    df['soil_x_rain']    = df['Soil_Moisture']  * df['Rainfall_mm'] / 1000.0
    df['temp_x_dryness'] = df['Temperature_C']  * (100 - df['Soil_Moisture']) / 100.0

RATIO_FEATURES = ['water_avail', 'heat_stress', 'rain_cool', 'wind_evap', 'soil_x_rain', 'temp_x_dryness']
NEW_NUMS += RATIO_FEATURES
print(f"Ratio features: {RATIO_FEATURES}")

In [ ]:
FEATURES = NUMS + CATS + NEW_NUMS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS
print(f'Total features: {len(FEATURES)}')

In [ ]:
TE_COLUMNS = NUM_AS_CAT + CATS + NEW_CATS
TO_REMOVE += NUM_AS_CAT + CATS + NEW_CATS

# Ordered Target Encoder

In [ ]:
from functools import reduce

class OrderedTE:
    def __init__(self, a=1):
        self.a = a

    def fit(self, train, category_cols=(), target_col='target'):
        self.category_cols = category_cols
        self.classes_ = sorted(train[target_col].unique())
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values
        self.stats_ = {}
        for c in category_cols:
            stats_list = []
            for k, cls in enumerate(self.classes_):
                y   = (train[target_col] == cls).astype(int)
                grp = train[[c]].assign(y=y.values)
                cum_cnt = grp.groupby(c, observed=False)['y'].cumcount()
                cum_sum = grp.groupby(c, observed=False)['y'].cumsum() - grp['y']
                prior   = self.global_prior_[k]
                te      = (cum_sum + self.a * prior) / (cum_cnt + self.a)
                train[f'{c}_TE_cls{cls}'] = te.values
                agg = grp.groupby(c, observed=False)['y'].agg(count='count', total='sum').reset_index()
                agg.columns = [c, f'{c}_n_{cls}', f'{c}_s_{cls}']
                stats_list.append(agg)
            self.stats_[c] = reduce(lambda l, r: l.merge(r, on=c, how='outer'), stats_list)
        return train

    def transform(self, test):
        for c in self.category_cols:
            test = test.merge(self.stats_[c], on=c, how='left')
            for k, cls in enumerate(self.classes_):
                n_col, s_col = f'{c}_n_{cls}', f'{c}_s_{cls}'
                prior = self.global_prior_[k]
                if n_col in test.columns:
                    test[f'{c}_TE_cls{cls}'] = ((test[s_col] + self.a * prior) / (test[n_col] + self.a)).fillna(prior)
                    test.drop(columns=[n_col, s_col], inplace=True)
                else:
                    test[f'{c}_TE_cls{cls}'] = prior
        return test

# XGB params — tuned for 4GB RTX 3050

In [ ]:
np.random.seed(11)

FOLDS = 5


In [ ]:
xgb_params = {
    'n_estimators':     50000,
    'max_depth':        4,
    'colsample_bytree': 0.8,
    'subsample':        0.8,
    'learning_rate':    0.05,
    'n_jobs':           -1,      # use all CPU cores
    'enable_categorical': True,
    'alpha':            5,
    'reg_lambda':       5,
    'gamma':            0.1,
    'max_leaves':       30,
    'min_child_weight': 2,
    'tree_method':      'hist',
    # device='cpu': with this many features + orig data the GPU has <14MB free
    # before XGBoost even starts. CPU hist with n_jobs=-1 is the right call here.
    'device':           'cpu',
    'objective':        'binary:logistic',
}


# Model Training — StratifiedKFold + Original Data in Training

In [ ]:
def metric(y_true, y_pred, sample_weight=None):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred, sample_weight=sample_weight)

def flush_gpu():
    """Free memory between models."""
    gc.collect()

In [ ]:
# %%time

print(f"\n{'='*80}")
print("TRAINING XGBOOST ONE-VERSUS-REST (with original data in training)")
print("="*80)

# StratifiedKFold: guarantees class balance in every fold
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=11)

orig_train = orig[FEATURES + [TARGET]].reset_index(drop=True).copy()

oof         = np.zeros((len(train), 3))
pred        = np.zeros((len(test),  3))
importances = {0: [], 1: [], 2: []}
metric_folds = []

for fold_i, (train_index, val_index) in enumerate(skf.split(train, train[TARGET])):
    print(f"\n{'='*60}")
    print(f"Fold {fold_i+1}/{FOLDS}")
    print('='*60)

    # ── Split ────────────────────────────────────────────────────
    X_train = train.loc[train_index, FEATURES + [TARGET]].reset_index(drop=True).copy()
    y_train_mc = train.loc[train_index, TARGET].values
    X_val      = train.loc[val_index,   FEATURES].reset_index(drop=True).copy()
    y_val_mc   = train.loc[val_index,   TARGET].values
    X_orig_te  = orig_train.copy()
    X_test     = test[FEATURES].reset_index(drop=True).copy()

    # ── Target encoding (fitted on train fold only — no leakage) ─
    te = OrderedTE(a=1)
    X_train = te.fit(X_train, category_cols=TE_COLUMNS, target_col=TARGET)
    X_val   = te.transform(X_val)
    X_test  = te.transform(X_test)
    X_orig_features = te.transform(X_orig_te.drop(TARGET, axis=1))
    y_orig_mc = X_orig_te[TARGET].values

    # ── Convert cats for XGBoost ─────────────────────────────────
    cat_cols = CATS + NEW_CATS + NUM_AS_CAT + NON_TE_CATS
    for df in [X_train, X_val, X_test, X_orig_features]:
        cols_here = [c for c in cat_cols if c in df.columns]
        df[cols_here] = df[cols_here].astype(str).astype('category')

    # Drop raw cat columns (replaced by TE)
    for df in [X_train, X_val, X_test, X_orig_features]:
        df.drop([c for c in TO_REMOVE if c in df.columns], axis=1, inplace=True)

    X_train = X_train.drop([TARGET], axis=1)
    COLS = X_train.columns

    # ── Concat train-fold + orig for model fitting ────────────────
    X_full = pd.concat([X_train, X_orig_features[COLS]], axis=0).reset_index(drop=True)
    y_full = np.concatenate([y_train_mc, y_orig_mc])

    # ── Train 3 binary OVR classifiers ───────────────────────────
    fold_oof  = np.zeros((len(X_val),  3))
    fold_pred = np.zeros((len(X_test), 3))

    for class_idx in range(3):
        print(f"\n  Binary classifier for class {class_idx}")

        y_bin_train = (y_full    == class_idx).astype(int)
        y_bin_val   = (y_val_mc  == class_idx).astype(int)
        s_wei       = compute_sample_weight('balanced', y_bin_train)

        model = xgb.XGBClassifier(
            **xgb_params,
            eval_metric='logloss',
            callbacks=[xgb.callback.EarlyStopping(rounds=300, metric_name='logloss', save_best=True)],
        )
        model.fit(
            X_full, y_bin_train,
            sample_weight=s_wei,
            eval_set=[(X_val, y_bin_val)],
            verbose=1000
        )

        fold_oof[:, class_idx]  = model.predict_proba(X_val)[:, 1]
        fold_pred[:, class_idx] = model.predict_proba(X_test[COLS])[:, 1]
        importances[class_idx].append(model.get_booster().get_score(importance_type='gain'))

        # Free GPU memory before next binary model (critical on 4GB VRAM)
        del model
        flush_gpu()

    # Normalise to sum-to-1
    fold_oof  /= fold_oof.sum(axis=1,  keepdims=True)
    fold_pred /= fold_pred.sum(axis=1, keepdims=True)

    oof[val_index] = fold_oof
    pred          += fold_pred

    bal_acc = metric(y_val_mc, fold_oof)
    metric_folds.append(bal_acc)
    print(f"\nFold {fold_i+1} Balanced Accuracy: {bal_acc:.5f}")

    del X_train, X_val, X_test, X_full, X_orig_features
    del y_train_mc, y_val_mc, y_full, y_orig_mc
    flush_gpu()

pred /= FOLDS

In [ ]:
print(f'Fold Balanced Accuracy {np.mean(metric_folds):.5f} +- {np.std(metric_folds):.5f}')
true = train[TARGET].values
print(f'Overall Balanced Accuracy: {metric(true, oof)}')

In [ ]:
print(f'\nUsed {len(COLS)} features:\n')
print(list(COLS))

# XGB Feature Importance (per class)

In [ ]:
all_importances = []
for class_idx in range(3):
    for fold_imp in importances[class_idx]:
        all_importances.append(fold_imp)

feature_names = all_importances[0].keys()
importances_mean = [
    np.mean([imp[feat] if feat in imp else 0 for imp in all_importances])
    for feat in feature_names
]

indices = np.argsort(importances_mean)
sorted_features   = np.array(list(feature_names))[indices][-60:]
sorted_importance = np.array(importances_mean)[indices][-60:]

plt.figure(figsize=(12, 18))
plt.barh(range(len(sorted_importance)), sorted_importance)
plt.yticks(range(len(sorted_features)), sorted_features)
plt.xlabel('Feature Importance (gain)')
plt.title('XGBoost Feature Importance — top 60')
plt.tight_layout()
plt.show()

# Post-processing: Optuna Class Weight Optimisation

In [ ]:
def accuracy_score(t, p):
    if len(p.shape) == 2:
        p = np.argmax(p, axis=1)
    return sum(np.sum((t == i) & (p == i)) / np.sum(t == i) for i in range(3)) / 3

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

oof_preds = oof
y = train[TARGET]

def objective(trial):
    cw = np.array([trial.suggest_float(f'cw{k}', 0.5, 3.0) for k in range(3)])
    adj = oof_preds * cw
    adj /= adj.sum(axis=1, keepdims=True)
    return accuracy_score(y, np.argmax(adj, axis=1))

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='class_weight_optimization'
)
study.optimize(objective, n_trials=500, show_progress_bar=True)

print(f"Best OOF balanced acc: {study.best_value:.6f}")
for k, name in enumerate(['Low', 'Medium', 'High']):
    print(f"  {name} weight = {study.best_params[f'cw{k}']:.4f}")

best_cw = np.array([study.best_params[f'cw{k}'] for k in range(3)])

final_oof  = oof * best_cw;  final_oof  /= final_oof.sum(axis=1,  keepdims=True)
final_pred = pred * best_cw; final_pred /= final_pred.sum(axis=1, keepdims=True)
final_preds = np.argmax(final_pred, axis=1)

print(f'\nFinal Overall Balanced Accuracy: {metric(true, final_oof)}')

# Save CSV

In [ ]:
# OOF and test predictions for ensembling
pd.DataFrame({'xgb': final_oof.flatten()}).to_csv(f'{NAME}_oof.csv',  index=False)
pd.DataFrame({'xgb': final_pred.flatten()}).to_csv(f'{NAME}_test.csv', index=False)
print("Saved oof and test predictions")

In [ ]:
sub = pd.DataFrame({'id': test_ids, TARGET: final_preds})
sub[TARGET] = sub[TARGET].map({0: 'Low', 1: 'Medium', 2: 'High'})
sub.to_csv(f'{NAME}_submission.csv', index=False)
print(f"Submission: {NAME}_submission.csv")
sub.head(10)